# Ranking Scorio Math models with `scorio.rank`

This notebook compares the four model configurations on a 10-question AIME 2026 window.
Ranking methods take an `L x M x N` tensor: models by questions by sampled attempts. Only
correctness and identity columns are read from the public Bucket. The comparison opens 40
remote Parquet files, so runtime depends on network latency.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

from scorio import rank

BUCKET_ROOT = "hf://buckets/harimo/scorio-math"


def pool_path(model, task, question_id):
    return f"{BUCKET_ROOT}/data/{model}/{task}/q{question_id:02d}.parquet"


def read_pools(model, task, question_ids, columns, max_workers=2):
    """Read selected columns from question files, preserving question order."""
    paths = [pool_path(model, task, q) for q in question_ids]

    def read_one(path):
        return pq.read_table(path, columns=columns)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        tables = list(executor.map(read_one, paths))
    return pa.concat_tables(tables).to_pandas()

models = ["Qwen3.6-35B-A3B", "gpt-oss-20b_low", "gpt-oss-20b_medium", "gpt-oss-20b_high"]
task = "aime_2026"
question_count = 10
question_ids = range(question_count)
columns = ["data_id", "seed", "evalscope_is_correct"]

matrices = []
for model in models:
    rows = read_pools(model, task, question_ids, columns).sort_values(["data_id", "seed"])
    assert rows.groupby("data_id").size().eq(80).all()
    matrices.append(rows.evalscope_is_correct.to_numpy().astype(int).reshape(question_count, 80))

R = np.stack(matrices)
print(R.shape, "models x questions x seeds")


(4, 10, 80) models x questions x seeds


## Bayes@N ranking

Rank 1 is best. `return_scores=True` also returns the values used to build the order.


In [2]:
ranks, scores = rank.bayes(R, return_scores=True)
leaderboard = pd.DataFrame({"rank": ranks.astype(int), "Bayes@N": scores}, index=models)
display(leaderboard.sort_values("rank").round(3))


,rank,Bayes@N
Qwen3.6-35B-A3B,1,0.945
gpt-oss-20b_high,2,0.934
gpt-oss-20b_medium,3,0.891
gpt-oss-20b_low,4,0.641


## Compare ranking methods

These methods summarize the same response tensor in different ways. `rasch_mml` handles
questions at the all-correct or all-incorrect boundary.


In [3]:
methods = ["bayes", "borda", "win_rate", "bradley_terry", "elo", "rasch_mml", "thompson"]
comparison = pd.DataFrame(
    {method: getattr(rank, method)(R).astype(int) for method in methods},
    index=models,
).sort_values("bayes")
display(comparison)


,bayes,borda,win_rate,bradley_terry,elo,rasch_mml,thompson
Qwen3.6-35B-A3B,1,1,1,1,1,1,1
gpt-oss-20b_high,2,2,2,2,2,2,2
gpt-oss-20b_medium,3,3,3,3,3,3,3
gpt-oss-20b_low,4,4,4,4,4,4,4


## Ranking at different sample budgets


In [4]:
budgets = [1, 2, 4, 8, 16, 32, 80]
sweep = pd.DataFrame(
    {f"n={n}": rank.bayes(R[:, :, :n]).astype(int) for n in budgets},
    index=models,
).sort_values("n=80")
display(sweep)
print("models whose rank changes between n=1 and n=80:",
      int((sweep["n=1"] != sweep["n=80"]).sum()), "of", len(models))


,n=1,n=2,n=4,n=8,n=16,n=32,n=80
Qwen3.6-35B-A3B,2,1,1,3,1,2,1
gpt-oss-20b_high,2,1,1,1,1,1,2
gpt-oss-20b_medium,1,1,1,1,3,3,3
gpt-oss-20b_low,4,4,4,4,4,4,4


models whose rank changes between n=1 and n=80: 2 of 4


The [ranking reference](https://github.com/mohsenhariri/scorio/blob/main/scorio/rank/README.md)
describes the voting, paired-comparison, item-response, graph, and listwise methods.
